# Your CV says 0.95. The leaderboard says 0.78.

That gap is usually not overfitting, and it is usually not your model. It is
that something in your feature matrix knows the answer — and knows it in a way
that will not exist when the model is scored.

The frustrating part is that the gap only shows up **after** you submit.

This notebook does three things:

1. plants a realistic leak in a real dataset and measures exactly how much it
   inflates a cross-validated score,
2. finds it in under a second **without training anything**,
3. runs the same check on a dataset where nobody planted anything, and finds
   seven columns whose *absence* carries the outcome.

Then there is a cell at the bottom you can point at your own competition data.

The tool is [`targetleak`](https://github.com/ShriyansBharuka/TargetLeak) —
Apache-2.0, no telemetry, your data never leaves this notebook.

In [ ]:
!pip install -q targetleak

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import targetleak

print("targetleak", targetleak.__version__)

## 1. A leak that looks exactly like a feature

The Adult census dataset, predicting whether income is over $50K. To it we add
one column of the shape that shows up in real pipelines all the time: a
**review note**, joined in from another system, that only gets filled in when
somebody was flagged as high-income.

Nobody adds a column called `the_answer`. They add `tax_review_note`, six
months after the label was defined, from a table whose timestamps nobody
checked. It has three perfectly innocent-looking values.

In [ ]:
from sklearn.datasets import fetch_openml

adult = fetch_openml("adult", version=2, as_frame=True, parser="pandas").frame
adult = adult.sample(12_000, random_state=0).reset_index(drop=True)
TARGET = "class"
y = (adult[TARGET].astype(str) == ">50K").astype(int)

rng = np.random.default_rng(0)
reviewed = rng.random(len(adult)) < 0.60

# Filled in for 60% of the high-income rows, and for 30% of the rest it says
# "no_review". Everything else is "pending". Note that it is NOT a copy of the
# target: 40% of high-income rows just say "pending" like everyone else.
adult["tax_review_note"] = np.where(
    (y == 1) & reviewed, "reviewed_high",
    np.where((y == 0) & (rng.random(len(adult)) < 0.30),
             "no_review", "pending"))

adult["tax_review_note"].value_counts()

## 2. What it costs you

Same model, same folds, same everything. The only difference is whether that
one column is in the frame.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score


def cv_auc(frame):
    X = frame.drop(columns=[TARGET]).copy()
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]):
            X[c] = X[c].astype("category")
    cats = [isinstance(X[c].dtype, pd.CategoricalDtype) for c in X.columns]
    model = HistGradientBoostingClassifier(max_iter=80, random_state=0,
                                           categorical_features=cats)
    return cross_val_score(model, X, y, cv=4, scoring="roc_auc").mean()


with_note = cv_auc(adult)
without = cv_auc(adult.drop(columns=["tax_review_note"]))

print(f"CV AUC with the joined column : {with_note:.4f}")
print(f"CV AUC without it             : {without:.4f}")
print(f"inflation                     : +{with_note - without:.4f}")

**About +0.06 of AUC, from one column, for free.**

In a competition that is the difference between a medal and the middle of the
pack — except it is not real, so it evaporates on data the leak does not cover.
That is your CV/LB gap, and cross-validation cannot see it: every fold has the
same leak in it, so every fold agrees.

Note that this leak is *not* a copy of the target. 40% of the high-income rows
say `pending` just like everyone else. A correlation check would call it a
decent feature and move on.

## 3. Finding it without training anything

The idea is cheap: **a model needs many features to reach AUC 0.95. A leak gets
there with one.** So a lone column that nearly solves the target is the prime
suspect, and testing that needs no model, no training run, and no labels beyond
the ones you already have.

In [ ]:
import time

t0 = time.time()
findings = targetleak.analyse(adult, target=TARGET)
print(f"analysed {adult.shape[0]:,} x {adult.shape[1]} "
      f"in {time.time() - t0:.1f}s, no model trained\n")

print(targetleak.report(findings, target=TARGET))

### Read that carefully, including what it did *not* say

It found the column, told you which value gives the answer away, how many rows
that covers, and how improbable that is against the base rate. Then it told you
what to do about it — naming a leak without saying how to fix it is a scolding,
not a tool.

But it reported a **warning**, not a critical, and the verdict says *"nothing
certain, things to confirm."* That is the right answer and it is worth
understanding why: 36% of the rows sit in pure groups, not 100%. A column that
partitions the whole target is the answer wearing a different hat. A column that
partitions part of it might be a leak or might be a genuinely strong feature,
and **the tool cannot know which** — only you know when `tax_review_note` gets
written.

Any tool that printed CRITICAL here would be guessing on your behalf. Which is
also why a clean report is not a clean bill of health: it points, you decide.

## 4. A real find, in a dataset nobody planted anything in

UCI's `cylinder-bands` — 540 rows of printing-press runs, predicting whether a
run came out defective. No synthetic leak, no author's thumb on the scale.

In [ ]:
bands = fetch_openml("cylinder-bands", version=2, as_frame=True,
                     parser="pandas").frame
print(bands.shape)

print(targetleak.report(targetleak.analyse(bands, target="band_type"),
                        target="band_type", show_code=False))

### Seven process measurements, one shared gap

At first glance that looks like a false-positive cluster. It is not. Those seven
columns share the same missing rows, and the target on those rows is not mixed:

In [ ]:
gap = bands["solvent_type"].isna()

print(f"rows where solvent_type is missing: {int(gap.sum())}")
print(bands.loc[gap, "band_type"].value_counts().to_string())
print("\nband_type across the whole dataset:")
print(bands["band_type"].value_counts(normalize=True).round(3).to_string())

**Every single one of those rows is a defective band, in a dataset where
defects are 42% of the total.**

The plain reading: when a defect occurred the run was abandoned and the
measurements were never taken. So the *absence* of a reading carries the
outcome. Train on this with a random split, impute the gaps with a column mean,
and you get a model that scores well and has learned nothing — it has learned
which rows were abandoned.

Note what this is not: none of those columns is individually predictive. Their
*values* are innocent. A correlation matrix, a feature-importance plot and a
`.describe()` all miss this completely, because the signal is in the NaN
pattern.

Whether that is a leak or a legitimate signal depends on when those
measurements get recorded relative to the defect being known — which a domain
expert can answer and no tool can. That is the honest shape of every finding
here.

## What it does not do

Worth stating plainly, because a leak detector that oversells itself is worse
than none:

- **A clean report is not proof of absence.** It cannot tell a leak from a
  genuinely easy problem.
- **It does not detect preprocessing leakage** — a scaler fit before the split
  lives in your code, not your data. Read your pipeline.
- **It cannot know your business.** A column that is legitimate at prediction
  time in one system is a leak in another.
- On the published benchmark it flags about **0.5% of clean columns** (3 of 628
  across 28 datasets), so expect a little noise and expect to suppress the odd
  column with `--ignore`.

Measured rather than asserted: **2/2 on documented leaks, and 91% of planted
leaks over 20 datasets - 20/20 when the leak is a clean copy of the answer**,
with controls that must stay quiet on data with nothing in it. The numbers and
the code that produces them are in the repo.

## Point it at your own data

Two lines. Add `split=` if you have a train/test flag in the frame, and
`group=` if rows belong to entities that must not straddle the split (a user, a
patient, a store, a session).

In [ ]:
# df = pd.read_csv("/kaggle/input/<your-competition>/train.csv")
# print(targetleak.report(targetleak.analyse(df, target="<your target>"),
#                         target="<your target>"))

# With a split column and an entity column:
# findings = targetleak.analyse(df, target="y", split="is_test", group="user_id")

# As a gate in a script - exit code 1 when anything critical is found:
#     targetleak train.csv --target y && python train.py

---

If it crashes on your data, or produces a finding that is obviously wrong,
that is the most useful thing you can report — and you do not need to share the
data, just the column types, the rough shape, and what it said.

[github.com/ShriyansBharuka/TargetLeak](https://github.com/ShriyansBharuka/TargetLeak)
· Apache-2.0